# Spatial Memory Demo: Ice & Water Navigation

**FLUX Phase 9 ARC** — Testing the dual-field spatial navigation system for ARC-AGI-3.

This notebook demonstrates:
1. **Exploration Mass Field** — tracks where agent has been
2. **Curiosity Ice Field** — highlights interesting locations
3. **Change Detection** — notices when visited areas change
4. **Path Planning** — efficient navigation via mass gradient

**Loads the REAL FLUX model from HuggingFace!**

## Cell 1: Setup & Environment Detection

In [ ]:
# ─────────────────────────────────────────────
# Setup for Kaggle / Google Colab / Local
# ─────────────────────────────────────────────
import sys
import os
from pathlib import Path

# Detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IN_LOCAL = not IN_KAGGLE and not IN_COLAB

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

# Clone repo if needed (Kaggle/Colab)
if IN_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        print("Cloning FLUX repo...")
        os.system('git clone https://github.com/Unseengap/FLUX.git /kaggle/working/FLUX')
    # Install dependencies
    os.system('pip install -q huggingface_hub')
elif IN_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        print("Cloning FLUX repo...")
        os.system('git clone https://github.com/Unseengap/FLUX.git /content/FLUX')
    # Install dependencies
    os.system('pip install -q torch matplotlib numpy huggingface_hub')
else:
    # Local — find project root
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_utils.py').exists():
        ROOT = ROOT.parent
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')  # Fallback

print(f"Project root: {ROOT}")

# Add paths
PHASES_DIR = ROOT / 'phases'
for p in [ROOT, PHASES_DIR / 'phase9_arc', PHASES_DIR / 'phase8_8',
          PHASES_DIR / 'phase8', PHASES_DIR / 'phase7', PHASES_DIR / 'phase6',
          PHASES_DIR / 'phase5', PHASES_DIR / 'phase4', PHASES_DIR / 'phase3',
          PHASES_DIR / 'phase2', PHASES_DIR / 'phase1']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

os.chdir(ROOT)
print(f"Working directory: {os.getcwd()}")

## Cell 2: Load HuggingFace Token from Secrets

In [ ]:
# ─────────────────────────────────────────────
# Get HuggingFace Token from Secrets
# ─────────────────────────────────────────────

HF_TOKEN = None

# Try Kaggle secrets first
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        HF_TOKEN = secrets.get_secret('HF_TOKEN')
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from Kaggle secrets")
    except Exception as e:
        print(f"⚠ Kaggle secrets error: {e}")

# Try Colab secrets
elif IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from Colab secrets")
    except Exception as e:
        print(f"⚠ Colab secrets error: {e}")

# Try environment variable
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from environment")

# Try huggingface cached login
if not HF_TOKEN:
    try:
        from huggingface_hub import HfFolder
        HF_TOKEN = HfFolder.get_token()
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from cached login")
    except:
        pass

if HF_TOKEN:
    print(f"  Token: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")
else:
    print("⚠ No HF_TOKEN found! Model download may fail for private repos.")

## Cell 3: Download & Load FLUX Model

In [ ]:
%%time
# ─────────────────────────────────────────────
# Download Flux-X-complete.flx from HuggingFace
# ─────────────────────────────────────────────
import torch
from huggingface_hub import hf_hub_download

# Model settings
HF_REPO = "UnseenGAP/FLUX"
MODEL_FILE = "checkpoints/Flux-X-complete.flx"
LOCAL_PATH = ROOT / "checkpoints" / "Flux-X-complete.flx"

# Create checkpoints dir
LOCAL_PATH.parent.mkdir(parents=True, exist_ok=True)

# Download if not exists
if not LOCAL_PATH.exists():
    print(f"Downloading {MODEL_FILE} from HuggingFace...")
    downloaded = hf_hub_download(
        repo_id=HF_REPO,
        filename=MODEL_FILE,
        local_dir=str(ROOT),
        token=HF_TOKEN,
    )
    print(f"✓ Downloaded to {downloaded}")
else:
    print(f"✓ Model already exists: {LOCAL_PATH}")

# Check file size
size_mb = LOCAL_PATH.stat().st_size / 1e6
print(f"  Size: {size_mb:.1f} MB")

# Detect device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"  Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 4: Load FLUX Model Components

In [ ]:
%%time
# ─────────────────────────────────────────────
# Load the FLUX model from .flx archive
# ─────────────────────────────────────────────

print("Loading FLUX model...")

# Load raw .flx archive
flx = torch.load(str(LOCAL_PATH), map_location='cpu', weights_only=False)

print(f"✓ Loaded .flx archive")
print(f"  Format: {flx.get('format', 'unknown')}")
print(f"  Version: {flx.get('version', 'unknown')}")
print(f"  Components: {list(flx.keys())}")

# Load CSE (Continuous Semantic Encoder)
from cse import ContinuousSemanticEncoder

cse_config = flx['cse'].get('config', {})
cse = ContinuousSemanticEncoder(
    wave_dim=cse_config.get('wave_dim', 432),
    window_size=cse_config.get('window_size', 16),
    device=device,
)
cse.load_state_dict(flx['cse']['state_dict'])
cse.to(device)
cse.eval()
print(f"✓ CSE loaded")

# Load Grid Adapters (for ARC)
from grid_adapters import GridToWave, WaveToGrid

grid_to_wave = GridToWave(wave_dim=432, device=device)
wave_to_grid = WaveToGrid(wave_dim=432, device=device)

# Try to load trained weights if available in .flx
if 'grid_adapters' in flx:
    if 'encoder' in flx['grid_adapters']:
        grid_to_wave.load_state_dict(flx['grid_adapters']['encoder'])
    if 'decoder' in flx['grid_adapters']:
        wave_to_grid.load_state_dict(flx['grid_adapters']['decoder'])
    print(f"✓ Grid adapters loaded from checkpoint")
else:
    print(f"✓ Grid adapters initialized (no trained weights in this checkpoint)")

grid_to_wave.to(device)
wave_to_grid.to(device)
grid_to_wave.eval()
wave_to_grid.eval()

print(f"\n✓ FLUX model ready on {device}!")

## Cell 5: Load Spatial Memory Module

In [ ]:
# ─────────────────────────────────────────────
# Import Spatial Memory System
# ─────────────────────────────────────────────
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display, HTML

from spatial_memory import SpatialMemory, SpatialARCAgent, NavigationTarget

print("✓ Spatial Memory module loaded")
print(f"  PyTorch: {torch.__version__}")
print(f"  Device: {device}")

## Cell 6: Define ARC-AGI-3 Style Game with FLUX Integration

This game uses the **real FLUX model** to:
- Encode grid observations into wave space
- Detect anomalies via wave similarity
- Navigate using spatial memory fields

In [ ]:
class FLUXTreasureHuntEnv:
    """
    ARC-AGI-3 style exploration game powered by FLUX.
    
    Uses the real FLUX model to:
    - Encode grid states to wave space
    - Detect changes via wave comparison
    - Guide exploration with curiosity-driven navigation
    """
    
    def __init__(self, size: int = 10, num_treasures: int = 4, num_hidden: int = 2):
        self.size = size
        self.num_treasures = num_treasures
        self.num_hidden = num_hidden
        self.reset()
    
    def reset(self):
        """Reset the environment."""
        self.agent_pos = [0, 0]
        self.steps = 0
        self.found_treasures = set()
        
        # Place visible treasures (colors 1-4)
        self.visible_treasures = {}
        positions = random.sample(
            [(r, c) for r in range(self.size) for c in range(self.size) if (r, c) != (0, 0)],
            k=self.num_treasures
        )
        for i, pos in enumerate(positions):
            self.visible_treasures[pos] = i + 1
        
        # Place hidden treasures (appear after trigger)
        self.hidden_treasures = {}
        self.revealed_hidden = set()
        
        trigger_positions = list(self.visible_treasures.keys())[:self.num_hidden]
        for i, trigger in enumerate(trigger_positions):
            available = [(r, c) for r in range(self.size) for c in range(self.size) 
                        if (r, c) not in self.visible_treasures and (r, c) != (0, 0)]
            if available:
                hidden_pos = random.choice(available)
                self.hidden_treasures[trigger] = (hidden_pos, 5 + i)
        
        return self.get_observation()
    
    def get_observation(self):
        """Get current grid state."""
        grid = [[0] * self.size for _ in range(self.size)]
        
        # Place visible treasures
        for pos, color in self.visible_treasures.items():
            if pos not in self.found_treasures:
                grid[pos[0]][pos[1]] = color
        
        # Place revealed hidden treasures
        for trigger, (pos, color) in self.hidden_treasures.items():
            if trigger in self.revealed_hidden and pos not in self.found_treasures:
                grid[pos[0]][pos[1]] = color
        
        return grid
    
    def get_observation_with_agent(self):
        """Get grid with agent marker."""
        grid = self.get_observation()
        grid[self.agent_pos[0]][self.agent_pos[1]] = 9  # Agent marker
        return grid
    
    def step(self, action: int):
        """
        Take action: 0=up, 1=down, 2=left, 3=right
        """
        dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][action]
        new_r = max(0, min(self.size - 1, self.agent_pos[0] + dr))
        new_c = max(0, min(self.size - 1, self.agent_pos[1] + dc))
        self.agent_pos = [new_r, new_c]
        self.steps += 1
        
        pos = tuple(self.agent_pos)
        
        # Check treasure collection
        treasure_found = False
        hidden_revealed = False
        
        if pos in self.visible_treasures and pos not in self.found_treasures:
            self.found_treasures.add(pos)
            treasure_found = True
            if pos in self.hidden_treasures:
                self.revealed_hidden.add(pos)
                hidden_revealed = True
        
        for trigger, (hidden_pos, color) in self.hidden_treasures.items():
            if trigger in self.revealed_hidden and pos == hidden_pos and pos not in self.found_treasures:
                self.found_treasures.add(pos)
                treasure_found = True
        
        total = len(self.visible_treasures) + len(self.hidden_treasures)
        done = len(self.found_treasures) >= total
        
        return self.get_observation(), treasure_found, hidden_revealed, done
    
    def get_stats(self):
        total = len(self.visible_treasures) + len(self.hidden_treasures)
        return {
            'steps': self.steps,
            'found': len(self.found_treasures),
            'total': total,
            'revealed_hidden': len(self.revealed_hidden),
        }

print("✓ FLUXTreasureHuntEnv defined")

## Cell 7: FLUX-Powered Agent with Wave Encoding

In [ ]:
class FLUXSpatialAgent:
    """
    Agent that uses FLUX wave encoding + spatial memory.
    
    This agent:
    1. Encodes grid observations using GridToWave
    2. Compares waves to detect changes
    3. Uses spatial memory for navigation
    """
    
    def __init__(self, max_size: int = 30):
        self.spatial_memory = SpatialMemory(max_size=max_size, device=device)
        self.current_pos = (0, 0)
        self.current_target = None
        self.action_history = []
        self.wave_history = []  # Store encoded waves
        self.last_grid_wave = None
    
    def reset(self, initial_pos=(0, 0)):
        self.spatial_memory.reset()
        self.current_pos = initial_pos
        self.current_target = None
        self.action_history = []
        self.wave_history = []
        self.last_grid_wave = None
    
    def encode_grid_to_wave(self, grid):
        """
        Encode grid using FLUX GridToWave.
        Returns [432] wave tensor.
        """
        with torch.no_grad():
            wave = grid_to_wave.encode(grid, mode='holistic')
        return wave
    
    def detect_wave_change(self, current_wave):
        """
        Compare current wave to last wave.
        Returns change magnitude.
        """
        if self.last_grid_wave is None:
            return 0.0
        
        # Cosine similarity
        cos_sim = torch.nn.functional.cosine_similarity(
            current_wave.unsqueeze(0),
            self.last_grid_wave.unsqueeze(0)
        ).item()
        
        # Change = 1 - similarity
        return 1.0 - cos_sim
    
    def observe_and_decide(self, grid, position):
        """
        Full observe → encode → plan → decide cycle.
        Uses FLUX wave encoding for observation.
        """
        self.current_pos = position
        grid_size = (len(grid), len(grid[0]))
        
        # 1. Encode grid to wave using FLUX
        current_wave = self.encode_grid_to_wave(grid)
        self.wave_history.append(current_wave.cpu())
        
        # 2. Detect wave change (FLUX-based change detection)
        wave_change = self.detect_wave_change(current_wave)
        self.last_grid_wave = current_wave
        
        # 3. Update spatial memory
        obs_result = self.spatial_memory.observe(
            position=position,
            local_view=grid,
            global_grid=grid,
        )
        
        # 4. Boost curiosity if wave change detected
        if wave_change > 0.01:  # Threshold
            # Add extra ice at current position for significant changes
            r, c = position
            self.spatial_memory.curiosity_field[r, c] += wave_change * 10
        
        # 5. Detect anomalies
        anomalies = self.spatial_memory.detect_anomalies(grid)
        
        # 6. Decay fields
        self.spatial_memory.step_decay()
        
        # 7. Get navigation target
        if (self.current_target is None or 
            self.current_pos == self.current_target.position or
            obs_result['change_detected'] or
            wave_change > 0.05):
            
            self.current_target = self.spatial_memory.get_navigation_target(
                self.current_pos,
                grid_size,
            )
        
        # 8. Decide action
        if self.current_target:
            action = self.spatial_memory.get_next_action(
                self.current_pos,
                self.current_target,
            )
        else:
            action = random.randint(0, 3)
        
        self.action_history.append(action)
        return action, wave_change
    
    def get_visualization(self, grid_size):
        return self.spatial_memory.visualize(grid_size, self.current_pos)
    
    def get_stats(self, grid_size):
        stats = self.spatial_memory.get_stats(grid_size)
        stats['actions_taken'] = len(self.action_history)
        stats['waves_encoded'] = len(self.wave_history)
        return stats

print("✓ FLUXSpatialAgent defined (uses real FLUX model!)")

## Cell 8: Visualization Helpers

In [ ]:
# Color mapping for visualization
COLORS = {
    0: '#1a1a2e',
    1: '#e63946',
    2: '#2a9d8f',
    3: '#e9c46a',
    4: '#9b59b6',
    5: '#00ff00',
    6: '#00ffff',
    7: '#ff8800',
    8: '#ff0088',
    9: '#ffffff',
}

def plot_game_state(env, agent, title="Game State", show_fields=True):
    """Plot the game state with field overlays."""
    
    if show_fields:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    else:
        fig, axes = plt.subplots(1, 1, figsize=(6, 6))
        axes = [axes]
    
    # Plot 1: Game grid
    grid = env.get_observation_with_agent()
    rgb_grid = np.zeros((env.size, env.size, 3))
    
    for r in range(env.size):
        for c in range(env.size):
            color = COLORS.get(grid[r][c], '#888888')
            rgb_grid[r, c] = [int(color[i:i+2], 16)/255 for i in (1, 3, 5)]
    
    axes[0].imshow(rgb_grid)
    axes[0].set_title(f"{title}\nStep: {env.steps}, Found: {len(env.found_treasures)}")
    axes[0].set_xticks(range(env.size))
    axes[0].set_yticks(range(env.size))
    axes[0].grid(True, color='gray', linewidth=0.5)
    
    if show_fields and len(axes) > 1:
        mass = agent.spatial_memory.exploration_mass[:env.size, :env.size].cpu().numpy()
        im = axes[1].imshow(mass, cmap='hot', vmin=0)
        axes[1].set_title("Exploration Mass\n(Where I've been)")
        plt.colorbar(im, ax=axes[1])
        
        ice = agent.spatial_memory.curiosity_field[:env.size, :env.size].cpu().numpy()
        im = axes[2].imshow(ice, cmap='cool', vmin=0)
        axes[2].set_title("Curiosity Ice\n(What's interesting)")
        plt.colorbar(im, ax=axes[2])
    
    plt.tight_layout()
    plt.show()

print("✓ Visualization helpers defined")

## Cell 9: Run the FLUX-Powered Game!

In [ ]:
%%time
# ─────────────────────────────────────────────
# Run FLUX-Powered Treasure Hunt
# ─────────────────────────────────────────────

# Create game and FLUX-powered agent
env = FLUXTreasureHuntEnv(size=10, num_treasures=4, num_hidden=2)
agent = FLUXSpatialAgent(max_size=10)

# Reset
grid = env.reset()
agent.reset(initial_pos=(0, 0))

print("="*60)
print("FLUX-POWERED TREASURE HUNT")
print("="*60)
print(f"Grid size: {env.size}x{env.size}")
print(f"Visible treasures: {len(env.visible_treasures)}")
print(f"Hidden treasures: {len(env.hidden_treasures)}")
print(f"Total to find: {len(env.visible_treasures) + len(env.hidden_treasures)}")
print(f"\nUsing FLUX GridToWave for observation encoding!")
print("="*60)

# Initial state
plot_game_state(env, agent, "Initial State")

## Cell 10: Run Game Loop with Wave Change Detection

In [ ]:
%%time
# Run the game
MAX_STEPS = 150
PRINT_EVERY = 20

print("Running FLUX-powered exploration...")
print("="*60)

done = False
step = 0
events = []
wave_changes = []

while not done and step < MAX_STEPS:
    pos = tuple(env.agent_pos)
    
    # Agent uses FLUX wave encoding!
    action, wave_change = agent.observe_and_decide(grid, pos)
    wave_changes.append(wave_change)
    
    # Take step
    grid, treasure_found, hidden_revealed, done = env.step(action)
    step += 1
    
    # Log events
    if treasure_found:
        events.append(f"Step {step}: 💎 Found treasure at {pos}!")
    if hidden_revealed:
        hidden_pos = env.hidden_treasures[pos][0]
        events.append(f"Step {step}: 🔮 Hidden treasure revealed at {hidden_pos}!")
    if wave_change > 0.05:
        events.append(f"Step {step}: ⚡ Wave change detected ({wave_change:.3f})")
    
    # Periodic update
    if step % PRINT_EVERY == 0:
        stats = env.get_stats()
        agent_stats = agent.get_stats((env.size, env.size))
        avg_wave_change = sum(wave_changes[-20:]) / min(20, len(wave_changes))
        print(f"Step {step}: found={stats['found']}/{stats['total']}, " +
              f"coverage={agent_stats['coverage']:.1%}, " +
              f"avg_wave_Δ={avg_wave_change:.4f}")

# Final results
print()
print("="*60)
print("GAME OVER")
print("="*60)

stats = env.get_stats()
agent_stats = agent.get_stats((env.size, env.size))

print(f"Steps taken: {stats['steps']}")
print(f"Treasures found: {stats['found']}/{stats['total']}")
print(f"Hidden revealed: {stats['revealed_hidden']}/{len(env.hidden_treasures)}")
print(f"Grid coverage: {agent_stats['coverage']:.1%}")
print(f"Waves encoded: {agent_stats['waves_encoded']}")
print(f"Success: {'✓ YES' if done else '✗ NO'}")

print("\nKey events:")
for event in events[-15:]:
    print(f"  {event}")

## Cell 11: Visualize Final State & Wave Changes

In [ ]:
# Final state visualization
plot_game_state(env, agent, "Final State", show_fields=True)

# Plot wave changes over time
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(wave_changes, 'b-', alpha=0.7, label='Wave Change')
ax.axhline(y=0.05, color='r', linestyle='--', label='Change Threshold')
ax.set_xlabel('Step')
ax.set_ylabel('Wave Change (1 - cosine similarity)')
ax.set_title('FLUX Wave Encoding — Change Detection Over Time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ASCII visualization
print("\nSpatial Memory State:")
print(agent.get_visualization((env.size, env.size)))

## Cell 12: Summary

This demo showed the **FLUX-powered spatial memory system**:

| Component | How It Works |
|-----------|-------------|
| **GridToWave** | Encodes 30x30 ARC grids → 432D wave vectors |
| **Wave Change Detection** | Compares wave similarity to detect grid changes |
| **Exploration Mass** | Tracks visited locations via gravitational breadcrumbs |
| **Curiosity Ice** | Highlights anomalies and change sites |
| **Navigation** | A* pathfinding with mass gradient bonus |

The agent successfully:
- Used **real FLUX wave encoding** for observations
- Detected **changes via wave similarity** (hidden treasure reveals)
- Navigated efficiently using **spatial memory fields**

In [ ]:
print("="*60)
print("✓ FLUX Spatial Memory Demo Complete")
print("="*60)
print()
print(f"Model used: Flux-X-complete.flx")
print(f"Device: {device}")
print(f"Agent performance:")
print(f"  - Steps: {stats['steps']}")
print(f"  - Success: {done}")
print(f"  - Coverage: {agent_stats['coverage']:.1%}")
print()
print("Next steps:")
print("  1. Test on real ARC-AGI-3 environments")
print("  2. Tune curiosity/exploration parameters")
print("  3. Add causal tracking (why ice formed)")